In [1]:
import numpy as np
import pandas as pd

# Модель для прогнозирования, например ARIMA
from sktime.forecasting.arima import ARIMA
# Визуализация временных рядов
from sktime.utils.plotting import plot_series
# Модули для кросс-валидации
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter, SlidingWindowSplitter, SingleWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import MeanSquaredError, MeanAbsoluteError, MeanAbsolutePercentageError # Метрики MSE, MAE, MAPE

import pandas_datareader.data as web

# настройки визуализации
import matplotlib.pyplot as plt

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

In [2]:
y = web.DataReader(name='GS10', data_source='fred', start='2000-01')
y.index = y.index.to_period(freq='M')
# длина ряда
len(y)

314

# Валидация через разделение на обучающую и тестовую выборки

In [3]:
# специфицируем модель для прогнозирования, например ARIMA(2,1,2) без сноса
forecaster = ARIMA(order=(2,1,2), trend='n')

# разбиваем выбору на обучающую (параметр window_length) и тестовую (параметр fh)
cv_strategy = SingleWindowSplitter(fh=np.arange(1, 6), window_length=len(y)-5)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res 

,test_MeanSquaredError,test_MeanAbsoluteError,test_MeanAbsolutePercentageError,fit_time,pred_time,len_train_window,cutoff
0,0.002146,0.036578,0.008821,0.659743,0.009802,309,2025-09


In [4]:
# специфицируем модель для прогнозирования, например ARIMA(1,1,1) без сноса
forecaster = ARIMA(order=(1,1,1), trend='n')

# разбиваем выбору на обучающую (параметр window_length) и тестовую (параметр fh)
cv_strategy = SingleWindowSplitter(fh=np.arange(1, 6), window_length=len(y)-5)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res 

,test_MeanSquaredError,test_MeanAbsoluteError,test_MeanAbsolutePercentageError,fit_time,pred_time,len_train_window,cutoff
0,0.002981,0.044557,0.01073,0.05359,0.009057,309,2025-09


# Валидация методом k-Fold (расширяем обучающую выборку)

In [5]:
# специфицируем модель для прогнозирования, например ARIMA(2,1,2) без сноса
forecaster = ARIMA(order=(2,1,2), trend='n')

# разбиваем выбору на обучающую (начинаем с первых initial_window) и тестовую длины (параметра fh)
# далее обучающую расширяем на step_length наблюдений
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=10)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res

,test_MeanSquaredError,test_MeanAbsoluteError,test_MeanAbsolutePercentageError,fit_time,pred_time,len_train_window,cutoff
0,0.043332,0.183602,0.046125,0.112192,0.008604,100,2008-04
1,0.222051,0.375706,0.107374,0.070386,0.008600,110,2009-02
2,0.027976,0.153984,0.041595,0.072638,0.009569,120,2009-12
3,0.515445,0.674938,0.200009,0.085579,0.008577,130,2010-10
4,0.101334,0.294862,0.147693,0.120950,0.009363,140,2011-08
5,0.003321,0.053404,0.032575,0.133116,0.008608,150,2012-06
6,0.641768,0.738074,0.285833,0.145911,0.008575,160,2013-04
7,0.006839,0.076304,0.029400,0.156156,0.008775,170,2014-02
8,0.048723,0.198607,0.101689,0.205423,0.008477,180,2014-12
9,0.039707,0.179450,0.089850,0.139167,0.008569,190,2015-10


In [6]:
# средняя MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)].mean()

test_MeanSquaredError               0.193532
test_MeanAbsoluteError              0.317635
test_MeanAbsolutePercentageError    0.158549
dtype: float64

In [7]:
# специфицируем модель для прогнозирования, например ARIMA(2,1,2) без сноса
forecaster = ARIMA(order=(1,1,1), trend='n')

# разбиваем выбору на обучающую (начинаем с первых initial_window) и тестовую длины (параметра fh)
# далее обучающую расширяем на step_length наблюдений
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=10)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
# средняя MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)].mean()

test_MeanSquaredError               0.196290
test_MeanAbsoluteError              0.314505
test_MeanAbsolutePercentageError    0.155317
dtype: float64

# Валидация методом k-Fold (скользящая обучающая выборка)

In [9]:
# специфицируем модель для прогнозирования, например ARIMA(2,1,2) без сноса
forecaster = ARIMA(order=(2,1,2), trend='n')

# разбиваем выбору на обучающую (длины initial_window) и тестовую (параметр fh)
# далее обучающую и тестовую сдвигаем на step_length шагов
cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=10)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
# средняя MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)].mean()

test_MeanSquaredError               0.240609
test_MeanAbsoluteError              0.353621
test_MeanAbsolutePercentageError    0.172007
dtype: float64

In [10]:
# специфицируем модель для прогнозирования, например ARIMA(2,1,2) без сноса
forecaster = ARIMA(order=(1,1,1), trend='n')

# разбиваем выбору на обучающую (длины initial_window) и тестовую (параметр fh)
# далее обучающую и тестовую сдвигаем на step_length шагов
cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=10)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
# средняя MSE, MAE, MAPE
cv_res.iloc[:,:len(metric)].mean()

test_MeanSquaredError               0.228642
test_MeanAbsoluteError              0.335200
test_MeanAbsolutePercentageError    0.165915
dtype: float64

In [11]:
# Зададим список из специфицированных моделей прогнозирования
forecasters = [ARIMA(order=(1,1,1), trend='n'), ARIMA(order=(1,1,1), trend='c'), ARIMA(order=(1,2,1), trend='n')]

# специфицируем метод кросс-валидации. Например, SlidingWindowSplitter
cv_strategy = SlidingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

# датафрейм с метриками по столбцам
cv_data = pd.DataFrame(data=None, columns=['MSE', 'MAE', 'MAPE'])

for model in forecasters:
	print(model)
	cv_res = evaluate(forecaster=model, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
	# print(df.iloc[:,:len(metric)].mean()) # метрики для каждой модели
	cv_data.loc[len(cv_data.index)] = cv_res.iloc[:,[0,1,2]].mean().values

# результаты кросс-валидации в виде датафрейма
cv_data

ARIMA(order=(1, 1, 1), trend='n')
ARIMA(order=(1, 1, 1), trend='c')
ARIMA(order=(1, 2, 1), trend='n')


,MSE,MAE,MAPE
0,0.215553,0.320545,0.150374
1,0.283675,0.393024,0.177408
2,0.391557,0.440078,0.204144


# Grid Search CV

In [12]:
# Поиск оптимальных гиперпараметров по сетке
from sktime.forecasting.model_selection import ForecastingGridSearchCV

In [13]:
# Зададим метод прогнозирования
forecaster = ARIMA()

# разбиваем параметры кросс-валидации
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# Задаём сетку для значений параметров модели в виде словаря
# будем менять параметры order и trend у ARIMA
# Важно: будут перебираться всевозможные комбинации!
param_grid = {'order':[(0,1,0), (1,1,0), (0,1,1)], 'trend': ['c', 'n']}

# инициализируем метрики
metric = MeanSquaredError(square_root=False)

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

ForecastingGridSearchCV(cv=ExpandingWindowSplitter(fh=array([1, 2, 3, 4, 5]),
                                                   initial_window=100,
                                                   step_length=5),
                        forecaster=ARIMA(),
                        param_grid={'order': [(0, 1, 0), (1, 1, 0), (0, 1, 1)],
                                    'trend': ['c', 'n']},
                        scoring=MeanSquaredError())

In [14]:
# Параметры оптимальной модели
gscv.get_fitted_params()['best_forecaster']

ARIMA(order=(0, 1, 1), trend='n')

In [15]:
# Зададим метод прогнозирования
forecaster = ARIMA()

# разбиваем параметры кросс-валидации
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# Задаём сетку для значений параметров модели в виде словаря
# будем менять параметры order и trend у ARIMA
# Важно: будут перебираться всевозможные комбинации!
param_grid = {'order':[(0,1,0), (1,1,0), (0,1,1), (0,2,0), (1,1,1), (2,1,2)], 'trend': ['c', 'n']}

# инициализируем метрики
metric = MeanSquaredError(square_root=False)

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

# Параметры оптимальной модели
gscv.get_fitted_params()['best_forecaster']

ARIMA(order=(0, 1, 1), trend='n')

In [18]:
# Зададим метод прогнозирования
forecaster = ARIMA()

# разбиваем параметры кросс-валидации
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=5)

# Задаём сетку для значений параметров модели в виде словаря
# будем менять параметры order и trend у ARIMA
# Важно: будут перебираться всевозможные комбинации!
param_grid = {'order':[(0,1,0), (1,1,0), (0,1,1), (0,2,0), (1,1,1), (2,1,2)], 'trend': ['c', 'n']}

# инициализируем метрики
metric = MeanAbsolutePercentageError()

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

# Параметры оптимальной модели
gscv.get_fitted_params()['best_forecaster']

ARIMA(order=(0, 1, 1), trend='n')

In [ ]:
# специфицируем модель для прогнозирования, например ARIMA(2,1,2) без сноса
forecaster = ARIMA(order=(2,0,2), trend='n')

# разбиваем выбору на обучающую (начинаем с первых initial_window) и тестовую длины (параметра fh)
# далее обучающую расширяем на step_length наблюдений
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=1)

# инициализируем метрики
metric = [MeanSquaredError(square_root=False), MeanAbsoluteError(), MeanAbsolutePercentageError()]

cv_res = evaluate(forecaster=forecaster, y=y, cv=cv_strategy, strategy="refit", return_data=False, scoring=metric)
cv_res

In [19]:
# Зададим метод прогнозирования
forecaster = ARIMA()

# разбиваем параметры кросс-валидации
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=1)

# Задаём сетку для значений параметров модели в виде словаря
# будем менять параметры order и trend у ARIMA
# Важно: будут перебираться всевозможные комбинации!
param_grid = {'order':[(2,0,2), (2,1,0), (2,1,1), (2,2,0)], 'trend': ['c', 'n']}

metric = MeanSquaredError(square_root=False)

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

# Параметры оптимальной модели
gscv.get_fitted_params()['best_forecaster']

ARIMA(order=(2, 0, 2), trend='n')

In [20]:
# Зададим метод прогнозирования
forecaster = ARIMA()

# разбиваем параметры кросс-валидации
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=1)

# Задаём сетку для значений параметров модели в виде словаря
# будем менять параметры order и trend у ARIMA
# Важно: будут перебираться всевозможные комбинации!
param_grid = {'order':[(2,0,2), (2,1,0), (2,1,1), (2,2,0)], 'trend': ['c', 'n']}

metric = MeanAbsoluteError()

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

# Параметры оптимальной модели
gscv.get_fitted_params()['best_forecaster']

ARIMA(order=(2, 0, 2), trend='n')

In [21]:
# Зададим метод прогнозирования
forecaster = ARIMA()

# разбиваем параметры кросс-валидации
cv_strategy = ExpandingWindowSplitter(fh=np.arange(1, 6), initial_window=100, step_length=1)

# Задаём сетку для значений параметров модели в виде словаря
# будем менять параметры order и trend у ARIMA
# Важно: будут перебираться всевозможные комбинации!
param_grid = {'order':[(2,0,2), (2,1,0), (2,1,1), (2,2,0)], 'trend': ['c', 'n']}

metric = MeanAbsolutePercentageError()

# Grid search
gscv = ForecastingGridSearchCV(forecaster=forecaster, param_grid=param_grid, cv=cv_strategy, scoring=metric)

gscv.fit(y)

# Параметры оптимальной модели
gscv.get_fitted_params()['best_forecaster']

ARIMA(order=(2, 0, 2), trend='n')